# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")
print(f"Authors: {[a['@id'] for a in metadata['author']]}")
print(f"License: {metadata['license']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will now inspect the record sets in the dataset using their `@id` fields. These are the entry points for loading data with `mlcroissant`.

> **Note:** All entities (record sets, fields, columns) are referenced by their `@id` values for consistency.

In [ ]:
# Examine available record sets and their fields
record_sets = dataset.metadata.record_sets

print("RecordSets found:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    • {field.name} (@id: {field.id}, type: {field.data_type})")
    print("  Columns:")
    for col in rs.columns:
        print(f"    • {col.name} (@id: {col.id}, source: {col.source})")
    print()
# Choose one record set for next steps
if record_sets:
    main_record_set_id = record_sets[0].id
    print(f"Main Record Set chosen: {main_record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all RecordSet IDs
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

# Select main record set for analysis
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"Main RecordSet columns: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, select a numeric field from the main record set using its `@id` (e.g., GAD-7 total score or PHQ-9 total score), filter by a threshold, normalize the values, and group results by a demographic attribute (such as 'level_of_education' or 'village').

In [ ]:
# Example EDA
main_rs = dataset.metadata.get_record_set(main_rs_id)
numeric_fields = [f for f in main_rs.fields if f.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']]

# Choose first numeric field for demonstration
numeric_field_id = numeric_fields[0].id if numeric_fields else None
group_field = None

# Try to select a group field (e.g. education or village)
possible_group_fields = [
    f for f in main_rs.fields if f.data_type in ['schema:Text'] and f.name.lower() in ['level_of_education', 'village', 'gender']
]
if possible_group_fields:
    group_field = possible_group_fields[0].id

df = dataframes[main_rs_id]

if numeric_field_id:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped data by {group_field} (showing mean {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of a numeric score (e.g., GAD-7 total) and its relationship with a demographic variable. All field and group names are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        # Boxplot by group
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides a rich set of data for exploration.
- Using `mlcroissant`, we accessed structured metadata and records via their `@id` fields, ensuring reproducible and precise data manipulation.
- Exploratory analysis and visualizations show variability in mental health scores and correlations with demographic attributes such as education and village.
- The schema-centric, FAIR approach eases downstream AI/ML workflows and supports robust data infrastructure for the African context.

For further analysis, refer to the Croissant schema and documentation, and try manipulating other record sets or fields using their `@id` values.